# 04 — ModernBERT come famiglia del testo (tentativo 09)

La famiglia **testo** di `02_modelli` è un TF-IDF. Qui la stessa famiglia letta da un **transformer** (ModernBERT-base,
solo il testo dell'articolo, come il TF-IDF), per vedere se nella combinazione con link e pagina porta qualcosa che
il TF-IDF non porta. Fissato il 22 ottobre, prima dei risultati.

**Ricetta**: quella delle Versioni 2–4 (1024 token, 2 epoche fisse, a ogni epoca al più 5 testate per gruppo a
rotazione e 20 articoli per testata, pesi di classe sulle testate pesate, batch effettivo 32, lr 2e-5, fp16; se la
memoria GPU finisce si riparte con batch dimezzato). Senza il 1/7 di gruppi tenuto da parte per la validazione:
con epoche fisse serviva solo al log.

**Per entrare nella combinazione** servono, come per le altre famiglie, le probabilità *fuori fold* delle testate di
training. Per ogni fold esterno quindi **4 addestramenti**:
- 3 **interni**, sulle stesse 3 parti per gruppo di `02_modelli` (`StratifiedGroupKFold(3)` con seed 42 sulle testate
  di training): si addestra su 2 parti e si predicono le testate della terza;
- 1 **finale**, su tutto il training: si predicono le testate valutate del test.

La probabilità di una testata è la **media delle probabilità dei suoi articoli** (al più 100 articoli per testata,
scelti a caso una volta per tutte, uguali per ogni seme). Si salva un file per addestramento: se il notebook si
interrompe, riparte da dove era. Prima di ogni addestramento si aspetta che sulla GPU (condivisa) ci siano almeno
`GB_GPU` liberi.

In [1]:
import gc
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

from comune import CLASSI, RESULTS, SEED, carica, metriche_di, registratore, testate

/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEME = 42                  # seme dell'addestramento (le divisioni in fold e in parti interne usano sempre SEED = 42)
MODELLO = "answerdotai/ModernBERT-base"
MAX_TOKEN = 1024
BATCH = 4                  # esempi per passo sulla GPU (come la Versione 4 sul server condiviso)
BATCH_EFFETTIVO = 32
BATCH_EVAL = 16
ARTICOLI_PER_TESTATA = 20  # per epoca, in addestramento
ARTICOLI_PREDETTI = 100    # al più, per testata, in predizione
MAX_TESTATE_PER_GRUPPO = 5
EPOCHE = 2
LR = 2e-5
WARMUP = 0.1
WEIGHT_DECAY = 0.01
FP16 = True
GB_GPU = 6.5               # memoria GPU libera necessaria per partire
FOLDS = 0                  # 0 = tutti
PROVA = False              # True: 40 testate, 1 fold, 1 epoca, risultati in Results/3-classificazione/prova/

In [3]:
# Parameters
SEME = 2


In [4]:
NOME = f"modernbert_s{SEME}"
base = RESULTS / "prova" if PROVA else RESULTS
OUT = base / "modernbert" / NOME
OUT.mkdir(parents=True, exist_ok=True)
registra = registratore(base / "modernbert" / f"avanzamento_{NOME}.log")
if PROVA:
    EPOCHE = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = {k: globals()[k] for k in ["SEME", "MODELLO", "MAX_TOKEN", "BATCH", "BATCH_EFFETTIVO", "ARTICOLI_PER_TESTATA", "ARTICOLI_PREDETTI",
                                    "MAX_TESTATE_PER_GRUPPO", "EPOCHE", "LR", "WARMUP", "WEIGHT_DECAY", "FP16", "PROVA"]}
config.update({"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"})
json.dump(config, open(OUT / "config.json", "w"), indent=1)
registra(f"=== {NOME}: {MODELLO}, {MAX_TOKEN} token, {EPOCHE} epoche, su {config['gpu']} ===")

art = carica(PROVA)
t = testate()
t = t[t.index.isin(set(art.domain))]           # in prova: solo le testate della prova (altrimenti sono tutte)
pos = {d: i for i, d in enumerate(t.index)}
rng0 = np.random.default_rng(SEED)
da_predire = {d: np.sort(rng0.choice(g.index.to_numpy(), min(len(g), ARTICOLI_PREDETTI), replace=False)) for d, g in art.groupby("domain")}
tok = AutoTokenizer.from_pretrained(MODELLO)
t0 = time.time()
ids = tok(art.text.tolist(), truncation=True, max_length=MAX_TOKEN)["input_ids"]
lunghezze = np.array([len(x) for x in ids])
PAD = tok.pad_token_id
registra(f"{len(art)} articoli di {len(t)} testate tokenizzati in {time.time() - t0:.0f} s; token: mediana {int(np.median(lunghezze))}")

2026-10-22 23:38:09  === modernbert_s2: answerdotai/ModernBERT-base, 1024 token, 2 epoche, su Tesla V100S-PCIE-32GB ===


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


2026-10-22 23:39:10  75394 articoli di 863 testate tokenizzati in 56 s; token: mediana 782


## Gli strumenti

In [5]:
def fissa_seme(s):
    s = int(s); random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def aspetta_gpu():
    while torch.cuda.is_available():
        libera = torch.cuda.mem_get_info()[0] / 2**30
        if libera >= GB_GPU:
            return
        registra(f"GPU occupata da altri: {libera:.1f} GB liberi, ne servono {GB_GPU}; riprovo tra 5 minuti")
        time.sleep(300)


def lotto(indici):
    L = max(lunghezze[i] for i in indici)
    x = np.full((len(indici), L), PAD, dtype=np.int64); m = np.zeros((len(indici), L), dtype=np.int64)
    for r, i in enumerate(indici):
        x[r, :lunghezze[i]] = ids[i]; m[r, :lunghezze[i]] = 1
    return torch.from_numpy(x).to(device), torch.from_numpy(m).to(device)


@torch.no_grad()
def predici(model, indici):
    model.eval()
    indici = np.asarray(indici); ordine = np.argsort(lunghezze[indici])
    out = np.zeros((len(indici), 5), dtype=np.float32)
    for s in range(0, len(indici), BATCH_EVAL):
        pezzo = ordine[s:s + BATCH_EVAL]
        x, m = lotto(indici[pezzo])
        with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
            out[pezzo] = model(input_ids=x, attention_mask=m).logits.float().cpu().numpy()
    return torch.softmax(torch.from_numpy(out), 1).numpy()


def addestra(domini, chiave):
    """Addestra ModernBERT sulle testate `domini` (la ricetta delle Versioni 2–4, epoche fisse)."""
    fissa_seme(SEME * 1000 + chiave)
    tr = art[art.domain.isin(set(domini))]
    articoli_di = {d: g.index.to_numpy() for d, g in tr.groupby("domain")}
    testate_di = {g: sorted(x.domain.unique()) for g, x in tr.groupby("gruppo")}
    t_tr = t.loc[sorted(articoli_di)]
    conta = t_tr.groupby("label").peso.sum().reindex(range(5), fill_value=0).to_numpy()
    pesi = torch.tensor(conta.sum() / (5 * np.maximum(conta, 1e-9)), dtype=torch.float32, device=device)

    def campione(epoca):
        rng = np.random.default_rng(SEME * 100000 + chiave * 100 + epoca)
        pezzi = []
        for g in sorted(testate_di):
            ds = testate_di[g]
            for d in (rng.choice(ds, MAX_TESTATE_PER_GRUPPO, replace=False) if len(ds) > MAX_TESTATE_PER_GRUPPO else ds):
                pezzi.append(rng.choice(articoli_di[d], min(len(articoli_di[d]), ARTICOLI_PER_TESTATA), replace=False))
        c = np.concatenate(pezzi); rng.shuffle(c)
        return c

    campioni = [campione(e) for e in range(1, EPOCHE + 1)]
    model = AutoModelForSequenceClassification.from_pretrained(MODELLO, num_labels=5, reference_compile=False).to(device)
    nodecay = ["bias", "LayerNorm.weight", "norm.weight"]
    gruppi = [{"params": [p for n, p in model.named_parameters() if not any(x in n for x in nodecay)], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in model.named_parameters() if any(x in n for x in nodecay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(gruppi, lr=LR)
    accumulo = max(1, BATCH_EFFETTIVO // BATCH)
    passi = sum(math.ceil(len(c) / (BATCH * accumulo)) for c in campioni)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * passi), passi)
    scaler = torch.amp.GradScaler("cuda", enabled=FP16)
    loss_fn = torch.nn.CrossEntropyLoss(weight=pesi)
    for epoca, camp in enumerate(campioni, 1):
        model.train(); tot, n = 0.0, 0; opt.zero_grad(); t0 = time.time()
        for s in range(0, len(camp), BATCH):
            pezzo = camp[s:s + BATCH]
            x, m = lotto(pezzo)
            y = torch.tensor(art.label.values[pezzo], device=device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
                loss = loss_fn(model(input_ids=x, attention_mask=m).logits.float(), y)
            scaler.scale(loss / accumulo).backward()
            tot += loss.item() * len(pezzo); n += len(pezzo)
            if (s // BATCH + 1) % accumulo == 0 or s + BATCH >= len(camp):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        registra(f"  epoca {epoca}/{EPOCHE}: {len(articoli_di)} testate, {n} articoli, loss {tot / n:.4f} ({time.time() - t0:.0f} s)")
    del opt, sched, scaler
    return model


def per_testata(model, domini):
    """Probabilità media dei (al più ARTICOLI_PREDETTI) articoli di ogni testata."""
    idx = np.concatenate([da_predire[d] for d in domini])
    P = predici(model, idx)
    righe = pd.DataFrame(P, columns=CLASSI); righe.insert(0, "domain", art.domain.values[idx])
    return righe.groupby("domain", sort=False)[CLASSI].mean().reset_index(), righe


def lavoro(nome, train, pred, chiave):
    """Un addestramento + predizione, salvato in OUT/<nome>.csv; riparte con batch dimezzato se la GPU non basta."""
    global BATCH, BATCH_EVAL
    if (OUT / f"{nome}.csv").exists():
        registra(f"{nome}: già fatto, salto"); return
    while True:
        aspetta_gpu()
        t0 = time.time(); finita = False
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        try:
            model = addestra(train, chiave)
            tab, righe = per_testata(model, pred)
            del model
        except RuntimeError as e:      # su artes4 NVML è rotto: la memoria finita può uscire come errore NVML/CUDA
            if not any(x in str(e) for x in ["out of memory", "CUBLAS_STATUS_ALLOC_FAILED", "CUDA error", "CUDACachingAllocator", "NVML"]):
                raise
            registra(f"{nome}: memoria GPU finita: {str(e)[:150]}")
            finita = True
        gc.collect(); torch.cuda.empty_cache()
        if not finita:
            break
        if BATCH == 1:
            raise RuntimeError("memoria GPU insufficiente anche con batch 1")
        BATCH, BATCH_EVAL = BATCH // 2, max(1, BATCH_EVAL // 2)
        registra(f"{nome}: riparto con batch {BATCH} (stesso batch effettivo {BATCH_EFFETTIVO})")
    if nome.endswith("finale"):
        righe.to_csv(OUT / f"{nome}_articoli.csv", index=False)
    tab.to_csv(OUT / f"{nome}.csv", index=False)       # per ultimo: la sua presenza vuol dire "fatto"
    gb = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0
    registra(f"{nome}: fatto in {(time.time() - t0) / 60:.0f} min ({len(train)} testate di training, {len(pred)} predette; memoria GPU massima {gb:.1f} GB)")

## I fold: 3 addestramenti interni (le stesse parti di `02_modelli`) e quello finale

In [6]:
folds = sorted(t.fold_gruppo.unique())
if PROVA:
    folds = folds[:1]
elif FOLDS:
    folds = [int(FOLDS)]
val = t.valutazione.to_numpy()
for f in folds:
    tr, te = np.where(t.fold_gruppo != f)[0], np.where(t.fold_gruppo == f)[0]
    tt = t.iloc[tr]
    for k, (a, b) in enumerate(StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(tt, tt.label, tt.gruppo)):
        registra(f">>> fold {f}, parte interna {k + 1}/3")
        lavoro(f"fold{f}_interno{k + 1}", list(t.index[tr[a]]), list(t.index[tr[b]]), f * 10 + k + 1)
    registra(f">>> fold {f}, finale")
    lavoro(f"fold{f}_finale", list(t.index[tr]), list(t.index[te[val[te]]]), f * 10)

2026-10-22 23:39:10  >>> fold 1, parte interna 1/3


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


2026-10-22 23:43:37    epoca 1/2: 289 testate, 3444 articoli, loss 1.5192 (263 s)


2026-10-22 23:47:40    epoca 2/2: 289 testate, 3444 articoli, loss 1.2798 (243 s)


...

2026-10-23 03:26:04  >>> fold 5, finale


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2026-10-23 03:32:35    epoca 1/2: 792 testate, 5216 articoli, loss 1.5158 (390 s)


2026-10-23 03:39:03    epoca 2/2: 792 testate, 5215 articoli, loss 1.3019 (388 s)


2026-10-23 03:40:01  fold5_finale: fatto in 14 min (792 testate di training, 59 predette; memoria GPU massima 5.1 GB)


## Il risultato di ModernBERT da solo (le testate valutate, probabilità media)

In [7]:
fatti = [f for f in folds if (OUT / f"fold{f}_finale.csv").exists()]
pr = pd.concat([pd.read_csv(OUT / f"fold{f}_finale.csv") for f in fatti]).set_index("domain")
y = t.label.reindex(pr.index).to_numpy()
m = metriche_di(y, pr[CLASSI].to_numpy().argmax(1))
registra(f"=== {NOME}: {len(fatti)} fold su {len(folds)}, {len(pr)} testate: F1 {m['f1_macro']:.3f}, binario {m['bin_accuratezza']:.3f} ===")
if len(fatti) == len(folds) and all((OUT / f"fold{f}_interno{k}.csv").exists() for f in folds for k in (1, 2, 3)):
    json.dump({k: v for k, v in m.items()}, open(OUT / "metriche.json", "w"), indent=1)
    (OUT / "FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))

2026-10-23 03:40:01  === modernbert_s2: 5 fold su 5, 301 testate: F1 0.332, binario 0.681 ===
